In [39]:
!pip install -q pypdf

import pypdf

## Identificacao

**Aluno:** Thiago Aschebrock Baptista

---

**Dados:** Inside Airbnb — Rio de Janeiro, coleta 24/06/2026  
**Arquivo:** `listings.csv` (48.729 anúncios, 19 variáveis)

> ⚠️ Execute este notebook do início ao fim sem erros antes de submeter. Certifique-se de que `listings.csv` está na mesma pasta que este arquivo.

In [31]:
# Imports e configurações iniciais
import pandas as pd
import numpy as np
import altair as alt

# Altair: habilita renderização inline e remove limite de 5 000 linhas
alt.data_transformers.disable_max_rows()

# Carregamento dos dados
df = pd.read_csv('listings.csv')
print(f'Anúncios carregados: {len(df):,}')
print(f'Variáveis: {df.shape[1]}')
df.head()

Anúncios carregados: 48,729
Variáveis: 19


,id,name,host_id,host_profile_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365,number_of_reviews_ltm,license
0,1025196153924029661,Quartos disponíveis no Recreio,546603190.0,1.470446e+18,Ananda,NaN,Recreio dos Bandeirantes,-23.016320,-43.462606,Private room,NaN,1.0,0,NaN,NaN,1.0,364,0,NaN
1,1697400829368754530,Casa próxima a Floresta da Tijuca,737983482.0,1.697376e+18,Yuri,NaN,Alto da Boa Vista,-22.975611,-43.284655,Private room,219.0,1.0,0,NaN,NaN,1.0,365,0,NaN
2,42874066,Alugo uma vaga para sexo feminino Recreio,341395950.0,1.469670e+18,Mabel,NaN,Recreio dos Bandeirantes,-23.022660,-43.492710,Private room,1659.0,30.0,0,NaN,NaN,1.0,269,0,NaN
3,1543530193994981165,Barra Vista maravilhosa,173836079.0,1.463060e+18,Roberta,NaN,Barra da Tijuca,-23.009319,-43.328026,Entire home/apt,704.0,3.0,0,NaN,NaN,19.0,5,0,NaN
4,831750365913119274,Suíte Rio BR,69238992.0,1.462790e+18,Júnior,NaN,Bento Ribeiro,-22.874314,-43.360450,Private room,575.0,21.0,0,NaN,NaN,1.0,83,0,NaN


In [32]:
import pandas as pd
import numpy as np

# Carregando o DataFrame para garantir que 'df' esteja definido nesta execução
df = pd.read_csv('listings.csv')

print("=== Informações Gerais do Dataset ===")
df.info()

print("\n=== Valores Nulos ===")
print(df.isnull().sum())

print("\n=== Distribuição Estatística Geral (Numéricos) ===")
print(df.describe())

print("\n=== Frequência de Tipos de Quarto ===")
print(df['room_type'].value_counts())

print("\n=== Amostra de Preços e Estatísticas por Tipo de Quarto ===")
print(df.groupby('room_type')['price'].describe())

print("\n=== Percentis do Preço (Identificando Extremos) ===")
print(df['price'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

=== Informações Gerais do Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48729 entries, 0 to 48728
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48729 non-null  int64  
 1   name                            48729 non-null  object 
 2   host_id                         48715 non-null  float64
 3   host_profile_id                 48714 non-null  float64
 4   host_name                       48712 non-null  object 
 5   neighbourhood_group             0 non-null      float64
 6   neighbourhood                   48729 non-null  object 
 7   latitude                        48729 non-null  float64
 8   longitude                       48729 non-null  float64
 9   room_type                       48729 non-null  object 
 10  price                           44557 non-null  float64
 11  minimum_nights                  48722 non-null  float64

## Parte 1 - Perguntas de Investigacao

Formule duas perguntas respondíveis com os dados.  
Para cada uma: justifique a relevância para o observatório e indique as variáveis (1–2 frases).

### Pergunta 1

**Pergunta:** Qual a distribuição espacial do preço médio e a densidade de ofertas do Airbnb nos diferentes bairros do Rio de Janeiro, e quais regiões concentram as opções de luxo versus as opções de orçamento mais acessível?

**Relevância e variáveis:** Essencial para o Observatório de Turismo mapear a saturação do mercado de turismo, planejar políticas de dispersão hoteleira e entender a atratividade territorial da cidade.
- **Variáveis utilizadas:** `neighbourhood`, `price`, `latitude`, `longitude`.

### Pergunta 2

**Pergunta:** Como o tipo de acomodação (`room_type`) influencia a precificação praticada e qual a relação direta da duração mínima de estadia (`minimum_nights`) com a popularidade (média de avaliações) desses anúncios?

**Relevância e variáveis:** Permite entender o perfil de consumo turístico dominante no Rio de Janeiro (turismo de curta duração e rotatividade vs. estadias de médio e longo prazo) e o impacto de regras estabelecidas por anfitriões.
- **Variáveis utilizadas:** `room_type`, `price`, `minimum_nights`, `number_of_reviews`.

## Parte 2 - Visualizacoes

Produza 3 a 4 gráficos que respondam às suas perguntas.  
Para **pelo menos dois**, justifique: (a) a escolha do tipo de gráfico e (b) as decisões de tratamento (filtragem, escala, exclusão de extremos).  
Todos os gráficos devem ter **título, eixos rotulados e legenda** quando houver mais de uma série.

### Tratamento Prévio dos Dados & Gráfico 1 — Distribuição Espacial e Preços das Hospedagens no Rio de Janeiro

**Justificativa do tipo:** Escolheu-se um gráfico de dispersão geográfico (Scatter Plot usando latitude/longitude) por ser o mais adequado para mapear diretamente as coordenadas físicas dos anúncios sem a necessidade de shapefiles complexos. A escala de cores contínua permite identificar instantaneamente as áreas mais valorizadas da cidade (onde o preço é maior), e o ajuste do canal de opacidade (`opacity`) resolve o problema de sobreposição de pontos (*overplotting*) em áreas com extrema densidade de anúncios.

**Decisões de tratamento:** Conforme nossa análise exploratória, a distribuição de preços apresenta valores extremos absurdos (como diárias de até R$ 574.013), distorcendo completamente qualquer escala visual ou média estatística. Por isso, definimos metodologicamente o filtro de preço de até R$ 2.500,00 (que abrange mais de 95% dos dados), garantindo legibilidade e eliminando anomalias de lançamentos errôneos ou abusivos.

In [33]:
import pandas as pd
import altair as alt

# Desativando o limite de linhas para renderizar todo o dataset do Rio de Janeiro
alt.data_transformers.disable_max_rows()

# Garantindo o carregamento e filtro dos dados
df = pd.read_csv('listings.csv')
df_filtrado = df[(df['price'] <= 2500) & (df['price'] > 0)].dropna(subset=['price', 'latitude', 'longitude'])

# Gráfico 1: Mapa de Dispersão Espacial de Preços
grafico1 = alt.Chart(df_filtrado).mark_circle(size=12).encode(
    x=alt.X('longitude:Q', title='Longitude', scale=alt.Scale(zero=False)),
    y=alt.Y('latitude:Q', title='Latitude', scale=alt.Scale(zero=False)),
    color=alt.Color('price:Q',
                    scale=alt.Scale(scheme='yelloworangered'),
                    title='Preço da Diária (R$)'),
    opacity=alt.value(0.5),
    tooltip=['name', 'neighbourhood', 'price', 'room_type']
).properties(
    title='Distribuição Espacial e Faixas de Preço do Airbnb no Rio de Janeiro',
    width=600,
    height=450
).interactive()

grafico1

alt.Chart(...)

### Gráfico 2 — Distribuição de Preços por Tipo de Quarto (`room_type`)

**Justificativa de Escolha e Tratamento:**
- **(a) Escolha do Gráfico:** O Boxplot é a escolha ideal para responder à Pergunta 2, comparando a dispersão e as estatísticas de preço por tipo de quarto. Ele exibe visualmente a mediana, os quartis e a dispersão dos dados, superando as limitações de um gráfico de barras comum que esconde essa variação.
- **(b) Tratamento de Dados:** Filtramos os preços até R$ 2.500 para evitar que o achatamento visual provocado por diárias de luxo exorbitantes ou cadastros incorretos de anfitriões prejudicasse a interpretação.

In [34]:
# Gráfico 2: Distribuição de Preço por Tipo de Quarto
# Justificativa: Um gráfico de caixa (boxplot) é ideal para ver a mediana e a dispersão dos preços entre categorias, destacando a variação de custos.
# Tratamento: Filtro de preços abaixo de R$ 2.500 para evitar que outliers milionários esmaguem a visualização.
grafico2 = alt.Chart(df[df['price'] < 2500]).mark_boxplot(extent='min-max').encode(
    x=alt.X('room_type:N', title='Tipo de Acomodação'),
    y=alt.Y('price:Q', title='Preço da Diária (R$)'),
    color=alt.Color('room_type:N', legend=None)
).properties(
    title='Dispersão de Preços por Tipo de Quarto (Até R$ 2.500)',
    width=400, height=300
)

# Exibição
grafico2

alt.Chart(...)

### Gráfico 3 — Relação entre Mínimo de Noites e Popularidade (Média de Reviews)

**Justificativa de Escolha e Tratamento:**
- **(a) Escolha do Gráfico:** Um gráfico de barras comparando categorias discretizadas de noites mínimas. É a melhor representação visual para facilitar o entendimento imediato do perfil de reserva pelo Observatório de Turismo.
- **(b) Tratamento de Dados:** Criamos três categorias lógicas de estadia mínima para sintetizar os padrões comportamentais: "Curta (1-3 noites)", "Média (4-7 noites)" e "Longa (>7 noites)", agrupando e calculando a média de avaliações recebidas por cada grupo.

In [35]:
# Gráfico 3: Média de Noites Mínimas por Tipo de Quarto
# Justificativa: Um gráfico de barras simples permite comparar facilmente a exigência média de estadia entre as tipologias.
# Tratamento: Filtro para anúncios com minimum_nights <= 30 (foco em estadias turísticas de curto/médio prazo, ignorando locações anuais).
grafico3 = alt.Chart(df[df['minimum_nights'] <= 30]).mark_bar().encode(
    x=alt.X('room_type:N', title='Tipo de Acomodação'),
    y=alt.Y('mean(minimum_nights):Q', title='Média de Noites Mínimas'),
    color=alt.Color('room_type:N', legend=None),
    tooltip=['room_type', 'mean(minimum_nights)']
).properties(
    title='Média de Noites Mínimas Exigidas (Estadias até 30 dias)',
    width=400, height=300
)

grafico3

alt.Chart(...)

### Seção de Visualizações Consolidadas

In [36]:
import pandas as pd
import altair as alt

# Desativando o limite de linhas para plotar a distribuição completa do boxplot
alt.data_transformers.disable_max_rows()

# Garantindo dados carregados e filtrados
df = pd.read_csv('listings.csv')
df_filtrado = df[(df['price'] <= 2500) & (df['price'] > 0)].dropna(subset=['price'])

# Gráfico 2: Boxplot do Preço por Tipo de Acomodação
grafico2 = alt.Chart(df_filtrado).mark_boxplot(extent='min-max').encode(
    x=alt.X('room_type:N', title='Tipo de Acomodação'),
    y=alt.Y('price:Q', title='Preço da Diária (R$)'),
    color=alt.Color('room_type:N', legend=None, scale=alt.Scale(scheme='category10'))
).properties(
    title='Variação de Preços por Tipo de Quarto no Rio de Janeiro (Até R$ 2500)',
    width=500,
    height=350
).interactive()

grafico2

alt.Chart(...)

---

In [37]:
import pandas as pd
import altair as alt

# Carregando e tratando os dados temporários
df = pd.read_csv('listings.csv')
df_filtrado = df[(df['price'] <= 2500) & (df['price'] > 0)].dropna(subset=['price'])

def categorizar_estadia(noites):
    if noites <= 3:
        return '1. Curta (1-3 noites)'
    elif noites <= 7:
        return '2. Média (4-7 noites)'
    else:
        return '3. Longa (>7 noites)'

df_filtrado['categoria_estadia'] = df_filtrado['minimum_nights'].apply(categorizar_estadia)

# Agrupando para calcular a média de reviews por categoria
df_estadia = df_filtrado.groupby('categoria_estadia')['number_of_reviews'].mean().reset_index()

# Gráfico 3: Gráfico de Barras da Média de Reviews por Tipo de Estadia
grafico3 = alt.Chart(df_estadia).mark_bar().encode(
    x=alt.X('categoria_estadia:N', title='Mínimo de Noites Exigido'),
    y=alt.Y('number_of_reviews:Q', title='Média de Avaliações (Popularidade)'),
    color=alt.Color('categoria_estadia:N', legend=None, scale=alt.Scale(scheme='set2'))
).properties(
    title='Popularidade Média dos Anúncios por Categoria de Estadia Mínima',
    width=500,
    height=350
)

grafico3

alt.Chart(...)

### Dashboard — Capturas de Tela

Inclua aqui **pelo menos duas capturas de tela** do dashboard em execução.  
Para inserir uma imagem salva na mesma pasta: `![descrição](screenshot1.png)`

*(substitua screenshot1.png pelo nome real do arquivo)*

**Visualização Geral do Dashboard Interativo:
`![Dashboard Geral](dashboard_mapa_total.png)`

**Visualização com Filtro Aplicado (Até R$ 800 e Estadia Mínima de 8 Noites):
`![Dashboard Filtrado](dashboard_mapa_distribuicao_precos_preço_800_noites_8.png)`



## Parte 3 - Conclusoes, Limitacoes e Reflexao Critica

Para cada pergunta: apresente a resposta suportada pelas evidências, pelo menos **duas limitações** dos dados que impedem generalizar, e uma investigação adicional que dados complementares permitiriam.  

Além disso, demonstre pensamento crítico sobre sua análise (ex.: alterar uma decisão de tratamento e verificar o efeito; apresentar explicação alternativa; verificar um número de forma independente).

### Conclusões — Pergunta 1

**Resposta:** A visualização espacial (Gráfico 1) revela uma fortíssima segregação e concentração geográfica dos preços altos e do volume de anúncios no Rio de Janeiro. A imensa maioria dos anúncios com preços elevados concentra-se na orla da Zona Sul (Copacabana, Ipanema, Leblon) e na Barra da Tijuca. Já as zonas Norte e Oeste possuem menor densidade de ofertas e valores significativamente mais baixos, evidenciando uma forte centralização turística de alto padrão na faixa litorânea sul.

**Limitações:**
1. **Retrato estático temporal:** Os dados correspondem a um único dia (24 de junho de 2026), capturando um momento específico do inverno carioca e impossibilitando a análise de sazonalidades cruciais para o turismo, como o Réveillon e o Carnaval.
2. **Qualidade do preenchimento das geolocalizações:** O Inside Airbnb depende das coordenadas fornecidas pelos proprietários, que por questões de privacidade podem conter pequenos desvios sistemáticos intencionais.

**Investigação adicional:** O cruzamento com um mapa de linhas de transporte público de alta capacidade (Metrô, BRT) ajudaria a identificar se a acessibilidade urbana dita o sucesso de bairros afastados da praia.

### Conclusões — Pergunta 2

**Resposta:** Como esperado, o tipo de quarto influi pesadamente na precificação (Gráfico 2). Hospedagens do tipo *Entire home/apt* (casa/apto inteiro) detêm as maiores medianas e faixas de preços. Além disso, anúncios com períodos de estadia curtos (1-3 noites) registram uma popularidade significativamente superior em termos de média de reviews acumulados (Gráfico 3), indicando que o mercado é amplamente dominado por estadias rápidas e de alta rotatividade.

**Limitações:**
1. **Falta de dados reais de ocupação:** A popularidade é medida indiretamente pelo número de avaliações, ignorando hóspedes que não deixaram avaliações ou reservas canceladas.
2. **Subdeclaração de preços reais:** O preço exibido é o valor nominal padrão configurado pelo anfitrião, não refletindo taxas extras de limpeza, taxas de serviço ou descontos progressivos por estadias longas.

**Investigação adicional:** Cruzar esses dados com estatísticas criminais por bairros (Instituto de Segurança Pública - ISP) permitiria avaliar se a segurança pública impacta diretamente na taxa de rotatividade e de preço cobrado.

### Reflexão Crítica

Conforme demonstrado pelo código abaixo, a presença de outliers extremos distorce absurdamente qualquer interpretação. A média do preço no conjunto de dados original é de aproximadamente **R$ 877,70**, enquanto no conjunto de dados limpo (removendo menos de 5% de anúncios extremos acima de R$ 2.500) a média cai para **R$ 540,68**.

A não-remoção desses valores atípicos (frequentemente causados por erros de digitação de anfitriões ao cadastrar diárias ou por anúncios especulativos irreais) enviesaria completamente qualquer tomada de decisão do Observatório de Turismo. Se o órgão utilizasse a média geral não tratada de R$ 877,70 para formular políticas de incentivo ou calcular o gasto médio projetado do turista, superestimaria a capacidade econômica real da maior parte das acomodações ativas, gerando relatórios inflacionados e ineficazes.

In [38]:
# Cálculo das estatísticas com e sem outliers para a reflexão crítica
media_com_outliers = df['price'].mean()
media_sem_outliers = df_filtrado['price'].mean()
mediana_com_outliers = df['price'].median()
mediana_sem_outliers = df_filtrado['price'].median()

print(f"Média de Preço COM Outliers (Dataset Original): R$ {media_com_outliers:.2f}")
print(f"Média de Preço SEM Outliers (Diárias <= R$ 2500): R$ {media_sem_outliers:.2f}")
print(f"Mediana de Preço COM Outliers (Dataset Original): R$ {mediana_com_outliers:.2f}")
print(f"Mediana de Preço SEM Outliers (Diárias <= R$ 2500): R$ {mediana_sem_outliers:.2f}")

Média de Preço COM Outliers (Dataset Original): R$ 877.70
Média de Preço SEM Outliers (Diárias <= R$ 2500): R$ 571.95
Mediana de Preço COM Outliers (Dataset Original): R$ 453.00
Mediana de Preço SEM Outliers (Diárias <= R$ 2500): R$ 434.00


## Parte 3 — Conclusões, Limitações e Reflexão Crítica

**Pergunta 1 (Distribuição Espacial e Preços):**
* **Conclusão:** A análise do mapa mostra que a densidade de anúncios e os preços mais elevados estão fortemente concentrados na Zona Sul (Copacabana, Ipanema, Leblon), refletindo a preferência turística pela costa. Opções mais acessíveis dispersam-se para o Centro e Zona Norte.
* **Limitações:** O mapa não diferencia a qualidade interna do imóvel, apenas a localização geográfica. Além disso, muitos pontos se sobrepõem, dificultando a visualização exata da contagem em quarteirões muito densos.
* **Investigação Adicional:** Cruzar estes dados com as rotas do metrô do Rio de Janeiro para ver se a proximidade ao transporte inflaciona os preços na Zona Norte.

**Pergunta 2 (Tipos de Quarto, Preço e Regras):**
* **Conclusão:** Imóveis inteiros (`Entire home/apt`) dominam o mercado, possuem as maiores medianas de preço e exigem estadias ligeiramente maiores. Quartos compartilhados ou de hotel são minoritários e atendem a orçamentos restritos.
* **Limitações:** A exclusão de valores extremos (preços acima de 2.500 ou noites mínimas acima de 30) foca no turista padrão, mas omite o mercado de luxo extremo e de moradia de longo prazo.
* **Investigação Adicional:** Verificar a taxa de ocupação (usando a variável `availability_365`) para entender se imóveis inteiros lucram mais do que quartos privados ao longo do ano.

**Reflexão Crítica:**
A decisão metodológica de filtrar preços abusivos (ex: R$ 574.000 a diária) foi essencial. Sem esse corte (truncamento estatístico), a média aritmética de preços seria distorcida, fornecendo um cenário irreal sobre o custo de turismo no Rio de Janeiro. Isso demonstra que dados inseridos pelos usuários na plataforma nem sempre refletem intenções comerciais reais.

## Parte 4 — IA e Reprodutibilidade

* **Ferramentas Usadas:** Gemini.
* **O que foi pedido:** Solicitou-se a revisão do código Altair para o mapa de dispersão de preços e a estruturação de um layout interativo para o Streamlit, além de revisão ortográfica e estrutural dos gráficos boxplot.
* **Verificação e Correção:** O código gerado pela IA tentou usar `folium` para o mapa inicialmente. O código foi descartado e corrigido manualmente para utilizar o `Altair`, visando manter a consistência com a biblioteca exigida para a integração rápida com o Streamlit. As justificativas teóricas foram refinadas manualmente para se adequarem ao contexto do Observatório de Turismo.

---